# 09 · RAG: corpus, fragmentos, índice y evaluación del retriever (T-11 y T-16, bloque M)

**Dónde:** `pipeline/src/rag/corpus.py` (corpus y chunking), `pipeline/src/rag/indexar.py` (embeddings + pgvector),
`api/_lib/rag.py` (consulta con umbral y fuentes), `pipeline/src/rag/evaluar.py` (métricas).
**Por qué:** el área de atención responde preguntas sobre tarifas y reglas del recibo; el RAG responde **solo** con
los documentos y dice cuándo no hay evidencia. **Cómo:** documentos → fragmentos de 900 caracteres con solape de
150 → embeddings Gemini de 768 dimensiones → `rag.buscar_fragmentos` (Top-k por coseno) → umbral de evidencia →
LLM que debe citar los fragmentos. **Datos:** tarifario CEA 2026-T3 (`api/_lib/tarifas_cea.csv`), reglas del
recibo portadas de Odoo y la documentación de los datos simulados. Sin datos personales.

In [ ]:
import sys, pathlib
raiz = pathlib.Path.cwd()
while not (raiz / "pipeline").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from pipeline.notebooks.graficas import dibujar, tabla

In [ ]:
import json
def resultado(modulo, clave):
    """Payload publicado por el evaluador (pipeline/artefactos/resultados/<modulo>.json) o None."""
    ruta = raiz / "pipeline" / "artefactos" / "resultados" / f"{modulo}.json"
    if not ruta.exists():
        return None
    return next((f["payload"] for f in json.loads(ruta.read_text(encoding="utf-8")) if f["clave"] == clave), None)

In [ ]:
from pipeline.src.rag.corpus import construir_corpus, TAMANO, SOLAPE
docs = construir_corpus()
resumen = pd.DataFrame([{"titulo": d.titulo, "tipo": d.tipo, "fragmentos": len(d.fragmentos),
                         "caracteres": len(d.texto)} for d in docs])
print(f"{len(docs)} documentos · {resumen['fragmentos'].sum()} fragmentos · tamaño {TAMANO}, solape {SOLAPE}")
resumen

## Distribución del tamaño de los fragmentos

In [ ]:
import matplotlib.pyplot as plt
largos = [len(f) for d in docs for f in d.fragmentos]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(largos, bins=15)
ax.set_title("Tamaño de los fragmentos del corpus")
ax.set_xlabel("Caracteres por fragmento"); ax.set_ylabel("Fragmentos")
plt.show()
print(f"Mediana {pd.Series(largos).median():.0f} caracteres; máximo {max(largos)} (límite {TAMANO}).")

## Ejemplo de fragmento (con metadatos que se guardan en `rag.fragmentos`)

In [ ]:
from pipeline.src.rag.indexar import filas
documentos, fragmentos = filas(docs)
ej = fragmentos[0]
print({k: v for k, v in ej.items() if k != "contenido"})
print(ej["contenido"][:600])

## Control de alucinaciones (diseño de `api/_lib/rag.py`)

1. Si ninguna similitud supera el umbral, `evidencia=false` y **no se llama al LLM**.
2. El LLM recibe solo los fragmentos numerados y debe responder `SIN_EVIDENCIA` si no bastan.
3. La salida se valida con Pydantic y debe citar al menos un fragmento existente; si no, se responde de forma extractiva.

In [ ]:
from api._lib.rag import UMBRAL_SIMILITUD, SIN_EVIDENCIA, responder

class EmbeddingFijo:            # solo para demostrar el flujo sin llaves: NO es una métrica
    def embeber(self, textos): return [[1.0] + [0.0] * 767 for _ in textos]

sin_fuentes = lambda v, k: [{"fragmento_id": "x", "documento_id": "y", "titulo": "otro tema", "contenido": "…", "similitud": 0.2}]
salida, detalle = responder("¿Quién ganó el mundial?", 5, EmbeddingFijo(), sin_fuentes, llm=None)
print(f"umbral = {UMBRAL_SIMILITUD}; evidencia = {salida.evidencia}; respuesta: {salida.respuesta}")

## Preguntas de evaluación (T-16a)

Borrador en `docs/eval/rag_preguntas_borrador.csv`; **solo las validadas por una persona** entran a la métrica.

In [ ]:
from pipeline.src.rag.evaluar import cargar_preguntas
todas = cargar_preguntas(incluir_sin_validar=True)
validadas = cargar_preguntas()
print(f"{len(todas)} preguntas en el borrador · {sum(not p['con_evidencia'] for p in todas)} sin evidencia esperada · {len(validadas)} validadas")

## Resultado: Recall@k, MRR y abstención (rag_eval)

In [ ]:
p = resultado("rag_eval", "recall_mrr")
if p:
    dibujar(p)
else:
    print("PENDIENTE: aún no se corre `python -m pipeline.src.rag.evaluar` (requiere el índice en rag.fragmentos y preguntas validadas).")

## Conclusiones

In [ ]:
print(f"1. El corpus tiene {len(docs)} documentos y {resumen['fragmentos'].sum()} fragmentos; cada fragmento guarda fuente, tipo y tamaño para citarlo.")
print(f"2. El umbral de evidencia ({UMBRAL_SIMILITUD}) hace que una pregunta fuera del corpus responda sin evidencia sin llamar al LLM.")
if p:
    print("3. " + p["conclusion"])
else:
    print(f"3. La métrica del retriever está pendiente: faltan el índice y la validación de {len(todas)} preguntas; no se reporta un número sin correrla.")
print("4. La app muestra las fuentes de cada respuesta (título, fragmento y similitud) en la página Asistente IA.")